In [43]:
import numpy as np
import pandas as pd
from glob import glob


clinical data file

In [28]:
# load original tcga clinical annotation data
tcga_clinical_info = pd.read_excel('./data/TCGA-CDR-SupplementalTableS1.xlsx', sheet_name='TCGA-CDR').iloc[:, 1:]

# get value counts for the unique type / histologies in that file and save to file
tcga_clinical_info.value_counts(['type', 'histological_type']).reset_index().sort_values(['type', 'count']).to_csv('./data/ncit_mapping.csv', index=False)

# use the file that is now annotated with ncit codes and labels
ncit_mapping = pd.read_csv('./data/ncit_mapping_2025-06-15.csv')
# create unique ids for tumor types
ncit_mapping['tumor_id'] = ncit_mapping['type'] + '::' + ncit_mapping['histological_type']
tcga_clinical_info['tumor_id'] = tcga_clinical_info['type'] + '::' + tcga_clinical_info['histological_type']
# merge ncit labels and code onto file
tcga_clinical_info = tcga_clinical_info.merge(ncit_mapping[['tumor_id', 'ncit_label', 'ncit_code']], on='tumor_id', how='left').drop(columns='tumor_id')


exome ngs mc3 data

In [31]:
maf = pd.read_csv(
    '/volumes/tcga-ngs/mc3.v0.2.8.PUBLIC.maf',
    sep='\t', low_memory=False, nrows=100
)

maf.columns.values
maf_columns = [
    'Chromosome', 'Start_Position', 'Reference_Allele', 'Tumor_Seq_Allele2', # uniquely define  mutation / variant
    'Hugo_Symbol', 'STRAND', 'Variant_Classification', 'Variant_Type', 'VARIANT_CLASS', 'HGVSp_Short', # metadata about mutation / variant
    'Tumor_Sample_Barcode', 't_alt_count', 't_depth', 'FILTER', # sample specific mutation / variant detection metrics
]

maf = pd.read_csv(
    '/volumes/tcga-ngs/mc3.v0.2.8.PUBLIC.maf',
    sep='\t', low_memory=False, usecols=maf_columns
)


In [ ]:
# get valid mutations and tally by tumor sample barcode
dna_samples = maf.loc[maf['FILTER'] == 'PASS', 'Tumor_Sample_Barcode'].value_counts().reset_index()
# extract patient and sample info
dna_samples['bcr_patient_barcode'] = dna_samples['Tumor_Sample_Barcode'].str.extract(r'^(TCGA-[^-]+-[^-]+)-')
dna_samples['bcr_sample_barcode'] = dna_samples['Tumor_Sample_Barcode'].str.extract(r'^(TCGA-[^-]+-[^-]+-\d+)')


Gene expression

In [68]:
# load rna expression data (only need sample names)
gene_exp = pd.read_csv('/volumes/tcga-ngs/EBPlusPlusAdjustPANCAN_IlluminaHiSeq_RNASeqV2.geneExp.tsv', sep='\t', low_memory=False, nrows=100)
rna_samples = pd.DataFrame(gene_exp.columns[1:].values, columns=['rna_sample'])
# extract patient and sample info
rna_samples['bcr_patient_barcode'] = rna_samples['rna_sample'].str.extract(r'^(TCGA-[^-]+-[^-]+)-')
rna_samples['bcr_sample_barcode'] = rna_samples['rna_sample'].str.extract(r'^(TCGA-[^-]+-[^-]+-\d+)')


Whole slide imaging files

In [ ]:
# find all TCGA svs files
wsi_samples = pd.DataFrame(glob('/volumes/tcga-wsi/*/TCGA*.svs'), columns=['wsi_path'])
# parse out info
wsi_samples['wsi_file'] = wsi_samples['wsi_path'].str.extract(r'/([^/]+)$')
wsi_samples['bcr_patient_barcode'] = wsi_samples['wsi_file'].str.extract(r'^(TCGA-[^-]+-[^-]+)-')
wsi_samples['bcr_sample_barcode'] = wsi_samples['wsi_file'].str.extract(r'^(TCGA-[^-]+-[^-]+-\d+)')


Merge samples, merge to clinical, and save

In [91]:
all_samples = dna_samples.drop(columns=['bcr_patient_barcode', 'count'])
all_samples = all_samples.merge(rna_samples[['bcr_sample_barcode', 'rna_sample']], on=['bcr_sample_barcode'], how='outer')
all_samples = all_samples.merge(wsi_samples[['bcr_sample_barcode', 'wsi_path']], on=['bcr_sample_barcode'], how='outer')
all_samples['bcr_patient_barcode'] = all_samples['bcr_sample_barcode'].str.extract(r'^(TCGA-[^-]+-[^-]+)-')
all_samples = all_samples[['bcr_patient_barcode', 'bcr_sample_barcode', 'Tumor_Sample_Barcode', 'rna_sample', 'wsi_path']]

In [ ]:
tcga_clinical_info = tcga_clinical_info.merge(all_samples, on='bcr_patient_barcode', how='left')
tcga_clinical_info.to_csv('./data/tcga_clinical_info.csv', index=False)
